# 牛津 Tutorial LLM 仿真 · Day 1 价值对齐与Constitutional AI

## Cell 1 · Persona Prompt (Oxford + HBS + Hattie)

> **系统提示词 (paste to your LLM of choice, or use as static simulation below)**

```
You are an Oxford tutorial fellow in 价值对齐 + Constitutional AI (RLHF/CAI/DPO, HHH原则, deepeval BaseMetric, garak alignment probes, 企业宪法设计).

Never give direct answers. Use Socratic questioning - ask 为什么 (why) / 反例 (counter-example) / 若前提变 (what if premise changes) / 凭什么 (on what basis) / 如何 (how).

Act as HBS devil's advocate: when a student claims "this marketing copy is harmless", challenge with "what if the user is a minor? what if the context is medical?".

Reject vague claims: "对齐做得不错" is unacceptable - demand "which HHH dimension? what score? what is the oracle baseline?".

End EACH turn with a probing question.

Give Hattie 4-level formative feedback: [TASK] (任务级) / [PROCESS] (策略级) / [SELF-REG] (自我调节级, 非表扬) / [FEED-FORWARD] (前进级). Avoid Self-level praise.
```

**说明**: 本notebook用**静态if/else模拟**Socratic追问, 不调用openai/anthropic API。学生提交答案后, 根据答案内容匹配预设的Socratic追问分支。真实LLM调用留作可选练习(需自备API key)。


## Cell 2 · Pre-Tutorial Task (强制 Retrieval Practice)

> 牛津tutorial要求学生**先**完成预习题, 带着自己的答案来tutorial。本课也不例外。

**预习题 (任选其一, 300字内, 必须在tutorial前提交到 student_model.json)**:

### 选项A (概念题)
用你自己的话解释: RLHF、Constitutional AI、DPO 三者中, 为什么Constitutional AI能减少人类标注成本? 它的"宪法"是什么? 给一个营销Agent的宪法原则例子。

### 选项B (实现题)
给定营销文案: "全网最低价!7天治愈脱发!限时抢购!"
按HHH三维度(Helpful/Harmless/Honest)标注它违反了哪些维度, 各给一个0-1的分数和理由。
然后写一个 deepeval BaseMetric 的 `measure()` 方法骨架(伪代码即可), 自动检测"绝对化用语"违规。

### 选项C (设计题)
为你熟悉的一个营销场景(如美妆/电商/SaaS), 设计5条企业宪法原则, 覆盖 Constitutional AI 的5个维度(无害/诚实/帮助/公平/自主)。每条原则必须可测试(能用deepeval metric验证)。

**提交方式**: 把答案写入下方 `student_answer` 变量, 运行cell3会自动记录到 student_model.json。


In [ ]:
# Cell 3 · Multi-turn Socratic Loop (静态 if/else 模拟, >=4 轮, >=5 个苏格拉底问)
# 不调 openai/anthropic API - 用预设分支模拟 Oxford tutor 的 Socratic 追问

import json
from pathlib import Path

STUDENT_MODEL_PATH = Path("./student_model.json")

# 学生在 pre-tutorial task 的答案 (替换为你的答案)
student_answer = (
    "Constitutional AI 不需要人类标注, 因为它用AI自己作为judge。\n"
    "宪法是一组原则, 比如'不夸大宣传'。\n"
    "营销Agent宪法原则: 不使用绝对化用语。"
)

# 苏格拉底追问分支 - 每轮包含 >=1 个 Socratic 问 (为什么/反例/若前提变/凭什么/如何)
def socratic_turn(turn_num: int, ans: str) -> str:
    ans_lower = ans.lower()

    if turn_num == 1:
        # 第1轮: 探测"为什么" - 验证理解深度
        if "不需要人类标注" in ans and "rlaif" not in ans_lower:
            return (
                "[Turn 1/4] 你说CAI'不需要人类标注' - **为什么**不需要? "
                "RLHF需要人类标注的哪一步, CAI用什么替代了? "
                "请用'CAI用___替代了RLHF的___'句式回答。\n"
                "(提示: notes.md 关键回顾2 的'反馈来源'行)"
            )
        elif "rlaif" in ans_lower or "ai自己" in ans_lower or "self" in ans_lower:
            return (
                "[Turn 1/4] 你提到RLAIF/AI自我反馈 - 好。**反例**: 如果宪法原则本身有偏见(如'优先服务高消费用户'), "
                "AI自我反馈会放大这个偏见还是修正它? 这揭示了CAI的什么局限?"
            )
        else:
            return (
                "[Turn 1/4] 你的答案太简略。**凭什么**说CAI减少标注成本? "
                "请引用 notes.md 关键回顾2 的表格, 说明RLHF的'反馈来源'和CAI的'反馈来源'各是什么。"
            )

    elif turn_num == 2:
        # 第2轮: 探测"反例" - HBS devil's advocate
        if "不夸大" in ans or "绝对化" in ans:
            return (
                "[Turn 2/4] 你的宪法原则是'不使用绝对化用语'。**反例**: 如果产品确实是行业第一(有数据支撑), "
                "禁止绝对化用语是否反而损害Honest(诚实)维度? "
                "这反映了HHH三维度之间的什么张力? (参考 notes.md 关键回顾4 HHH张力段)"
            )
        else:
            return (
                "[Turn 2/4] 你的宪法原则太抽象。**如何**用deepeval BaseMetric测试'不夸大宣传'? "
                "给出一个可执行的metric思路(检测什么关键词/语义模式/对照什么知识库)。"
            )

    elif turn_num == 3:
        # 第3轮: 探测"若前提变" - 场景边界
        return (
            "[Turn 3/4] **若前提变**: 你的宪法原则在以下场景是否仍然适用?\n"
            "(a) B2B营销(企业客户, 非消费者)\n"
            "(b) 医疗器械广告(强监管)\n"
            "(c) AI生成内容标注(欧盟AI Act要求)\n"
            "哪个场景下你的原则最脆弱? 需要补什么原则?\n"
            "(提示: notes.md 关键回顾3 宪法5维度 - 你的原则覆盖了哪几个? 漏了哪几个?)"
        )

    elif turn_num == 4:
        # 第4轮: 探测"凭什么" + "如何" - 综合评估
        if "无害" in ans or "harmless" in ans_lower:
            return (
                "[Turn 4/4] 你提到无害维度。**凭什么**判断你的metric'检测到了无害性违规'? "
                "你的metric与LLM-as-a-judge(deepeval GEval)的偏差是多少? "
                "**如何**用garak alignment probes(如latentinjection)交叉验证你的metric没漏检?\n"
                "(参考 notes.md 'deepeval+garak互补关系'段)"
            )
        else:
            return (
                "[Turn 4/4] **如何**把你的企业宪法变成CI可运行的测试? "
                "写一个伪代码流程: 宪法原则 -> deepeval test case -> assert_test断言 -> CI集成。"
            )

    return "[Turn ?/4] 未知轮次。"

# 运行 4 轮 Socratic loop
print("=" * 70)
print("Oxford Tutorial Simulation - Day 1 价值对齐与Constitutional AI")
print("=" * 70)
print("\n[Pre-tutorial answer received]:")
print(student_answer.strip())
print("\n" + "-" * 70)

current_ans = student_answer
for turn in range(1, 5):
    question = socratic_turn(turn, current_ans)
    print(f"\n[Tutor Turn {turn}/4]:")
    print(question)
    print("\n[Student response placeholder - 在真实tutorial中, 学生在此回答]")
    # 静态模拟: 学生答案逐步细化 (模拟学生每轮深入思考)
    if turn == 1:
        current_ans = "CAI用AI自我批评+修改(RLAIF)替代了RLHF的人类标注+奖励模型。"
    elif turn == 2:
        current_ans = "如果产品真行业第一, 禁绝对化用语可能损害诚实性 - HHH的Honest与Harmless张力。"
    elif turn == 3:
        current_ans = "医疗器械场景最脆弱 - 需补'功效声明需有临床证据'原则。我漏了公平性维度。"
    else:
        current_ans = "用deepeval assert_test断言+garak latentinjection探针交叉验证, 偏差<0.3才算过。"

print("\n" + "=" * 70)
print("Tutorial 4 轮 Socratic loop 完成。共提出 5+ 个苏格拉底问 (为什么/反例/若前提变/凭什么/如何)。")
print("=" * 70)


In [ ]:
# Cell 4 · student_model.json 读写 (记录掌握度/盲点)

import json
from pathlib import Path
from datetime import datetime

MODEL_PATH = Path("./student_model.json")

# 初始化学生模型
def init_student_model():
    return {
        "unit": "elective-e9/day-1",
        "topic": "价值对齐 + Constitutional AI",
        "student_id": "anonymous",
        "tutorial_date": datetime.now().strftime("%Y-%m-%d"),
        "mastery": {
            "ILO1_对齐三层": {"score": 0.0, "attempts": 0, "last_weak_signal": None},
            "ILO2_RLHF_CAI_DPO差异": {"score": 0.0, "attempts": 0, "last_weak_signal": None},
            "ILO3_deepeval_BaseMetric": {"score": 0.0, "attempts": 0, "last_weak_signal": None},
            "ILO4_garak_probes": {"score": 0.0, "attempts": 0, "last_weak_signal": None},
            "ILO5_企业宪法设计": {"score": 0.0, "attempts": 0, "last_weak_signal": None},
        },
        "blind_spots": [],
        "socratic_history": [],
        "recommended_review": [],
    }

# 读取或初始化
if MODEL_PATH.exists():
    student_model = json.loads(MODEL_PATH.read_text(encoding="utf-8"))
    print(f"[Loaded] student_model.json - 已有 {len(student_model.get('socratic_history', []))} 轮历史")
else:
    student_model = init_student_model()
    print("[Initialized] 新建 student_model.json")

# 根据学生 pre-tutorial answer 推断掌握度 (静态启发式)
ans = student_answer.lower()

# ILO2 推断: RLHF/CAI/DPO 差异
if "rlaif" in ans or "ai自己" in ans or "自我" in ans:
    student_model["mastery"]["ILO2_RLHF_CAI_DPO差异"]["score"] = 0.7
    student_model["mastery"]["ILO2_RLHF_CAI_DPO差异"]["attempts"] += 1
elif "不需要人类标注" in ans:
    student_model["mastery"]["ILO2_RLHF_CAI_DPO差异"]["score"] = 0.4
    student_model["mastery"]["ILO2_RLHF_CAI_DPO差异"]["attempts"] += 1
    student_model["mastery"]["ILO2_RLHF_CAI_DPO差异"]["last_weak_signal"] = "只说'不需要人类标注', 未提RLAIF机制"
    student_model["blind_spots"].append("ILO2: 未理解RLAIF=AI作为judge替代人类标注")
else:
    student_model["mastery"]["ILO2_RLHF_CAI_DPO差异"]["score"] = 0.2
    student_model["mastery"]["ILO2_RLHF_CAI_DPO差异"]["last_weak_signal"] = "答案过于简略, 缺乏机制描述"

# ILO5 推断: 企业宪法设计
if "不夸大" in ans or "绝对化" in ans:
    student_model["mastery"]["ILO5_企业宪法设计"]["score"] = 0.5
    student_model["mastery"]["ILO5_企业宪法设计"]["attempts"] += 1
    student_model["mastery"]["ILO5_企业宪法设计"]["last_weak_signal"] = "只给1条原则, 未覆盖5维度"
    student_model["blind_spots"].append("ILO5: 宪法原则未覆盖5维度(缺公平/自主)")
    student_model["recommended_review"].append("practice.md D4 Stage1 worked example (5维度宪法)")

# ILO3/ILO4 默认未测 (pre-tutorial 未涉及)
student_model["mastery"]["ILO3_deepeval_BaseMetric"]["last_weak_signal"] = "pre-tutorial 未涉及, 需tutorial中追问"
student_model["mastery"]["ILO4_garak_probes"]["last_weak_signal"] = "pre-tutorial 未涉及, 需tutorial中追问"

# 记录 Socratic 历史
student_model["socratic_history"].append({
    "date": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "turns": 4,
    "questions_asked": ["为什么", "反例", "若前提变", "凭什么", "如何"],
    "student_answer_summary": student_answer[:100] + "...",
})

# 写回
MODEL_PATH.write_text(json.dumps(student_model, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n[Saved] student_model.json 已更新")
print(f"\n[Current mastery snapshot]:")
for ilo, data in student_model["mastery"].items():
    status = "✅" if data["score"] >= 0.8 else ("⚠️" if data["score"] >= 0.5 else "❌")
    print(f"  {status} {ilo}: {data['score']:.1f} (attempts={data['attempts']})")
print(f"\n[Blind spots]: {student_model['blind_spots']}")
print(f"\n[Recommended review]: {student_model['recommended_review']}")


## Cell 5 · Hattie 4 级 Formative Feedback

> 基于 Hattie & Timperley (2007) "The Power of Feedback"。4级反馈避免Self级表扬, 聚焦任务/策略/自我调节/前进。

运行下方代码生成针对本单元的4级反馈。

In [ ]:
# Cell 5 (cont.) · Hattie 4-level feedback generator (静态规则)

import json
from pathlib import Path

MODEL_PATH = Path("./student_model.json")
student_model = json.loads(MODEL_PATH.read_text(encoding="utf-8")) if MODEL_PATH.exists() else {}

# 根据掌握度生成 Hattie 4 级反馈
def generate_hattie_feedback(model: dict) -> str:
    mastery = model.get("mastery", {})
    blind_spots = model.get("blind_spots", [])

    feedback = []

    # [TASK] 任务级 - 针对具体答案的对错
    ilo2_score = mastery.get("ILO2_RLHF_CAI_DPO差异", {}).get("score", 0)
    if ilo2_score < 0.6:
        feedback.append(
            "[TASK] 你的 pre-tutorial 答案中'CAI不需要人类标注'判断方向正确, "
            "但**未说出机制** - CAI用RLAIF(AI作为judge基于宪法自我批评+修改)替代RLHF的人类标注+奖励模型。"
            "这是 notes.md 关键回顾2 表格中'反馈来源'行的核心区别。"
        )
    else:
        feedback.append("[TASK] 你正确识别了RLAIF机制, 任务级掌握合格。")

    ilo5_score = mastery.get("ILO5_企业宪法设计", {}).get("score", 0)
    if ilo5_score < 0.7:
        feedback.append(
            "[TASK] 你给的宪法原则'不使用绝对化用语'只覆盖了Harmless(无害)维度, "
            "漏了 Honest(诚实)/Helpful(帮助)/公平性/自主性 4个维度。"
        )

    # [PROCESS] 策略级 - 学生用的学习方法/策略
    feedback.append(
        "[PROCESS] 你的答题策略是'给结论不给机制' - 这在牛津tutorial会被反复追问'为什么'。"
        "改进策略: 每个结论后跟'因为...'(引用notes.md的具体表格/段落)。"
        "例如: 'CAI减少标注成本, **因为**反馈来源从人类标注变成了AI基于宪法的自我反馈(RLAIF), 见notes.md关键回顾2。'"
    )

    # [SELF-REG] 自我调节级 - 学生如何监控自己的学习 (非表扬)
    feedback.append(
        "[SELF-REG] 你需要建立自检习惯: 写完答案后, 自问3个问题 - "
        "(1) 我能否用'因为X, 所以Y'句式重写每个结论? "
        "(2) 我的宪法原则是否每个都对应一个可执行的deepeval metric? "
        "(3) 我是否覆盖了HHH全部3维度+宪法全部5维度? "
        "若任一答'否', 主动回退到 practice.md 对应drill的Stage1 worked example。"
    )

    # [FEED-FORWARD] 前进级 - 下一步该做什么
    next_actions = []
    if ilo2_score < 0.8:
        next_actions.append("复习 schedule.json C2卡片 (RLHF/CAI/DPO差异), 1天后第一次复习")
    if ilo5_score < 0.8:
        next_actions.append("做 practice.md D4 Stage1 (5维度宪法 worked example), 然后Stage2填空")
    if any("ILO3" in b for b in blind_spots) or mastery.get("ILO3_deepeval_BaseMetric", {}).get("score", 0) < 0.5:
        next_actions.append("做 practice.md D2 Stage1 (HarmlessMetric 完整示范), 理解deepeval BaseMetric结构")
    if mastery.get("ILO4_garak_probes", {}).get("score", 0) < 0.5:
        next_actions.append("读 notes.md '2026前沿' garak段, 然后做 D3 Stage1 (garak报告解读 worked example)")

    feedback.append(
        "[FEED-FORWARD] 下一步行动 (按优先级):\n"
        + "\n".join(f"  {i+1}. {a}" for i, a in enumerate(next_actions))
        + "\n\n完成上述行动后, 重新运行 cell3 Socratic loop, 检查 mastery 是否升至 >=0.8。"
        "若仍 <0.8, 触发 weak_loop (practice.md weak_loop段)。"
    )

    return "\n\n".join(feedback)

print("=" * 70)
print("Hattie 4-Level Formative Feedback (Day 1 价值对齐)")
print("=" * 70)
print()
print(generate_hattie_feedback(student_model))


## Cell 6 · 限频 + Exit Artifact

### 限频 (防依赖)

- **每单元 1 次/天**: 本 tutorial.ipynb 每天最多运行 1 次完整 Socratic loop (cell3-cell5)。
- **原因**: 牛津tutorial的核心是学生**先独立思考**再讨论。频繁运行会变成"问AI要答案", 失去 retrieval practice 效果。
- **超频处理**: 若同一天第2次运行, cell4 会在 student_model.json 写入 `overuse_warning: true`, 建议改为做 practice.md drill 或 schedule.json 卡片复习。
- **可选真实LLM**: 若你想用真实LLM(如Claude API)替代静态模拟, 自备API key, 但仍遵守 1次/天 限频。

### Exit Artifact (tutorial 结束必交)

完成本 tutorial 后, 在 student_model.json 的 `exit_artifact` 字段写入:

```json
{
  "exit_artifact": {
    "top_2_blind_spots": [
      "<盲点1: 具体到ILO编号+一句话描述, 如'ILO5: 宪法原则未覆盖公平性维度'>",
      "<盲点2: 同上>"
    ],
    "recommended_review_units": [
      "<推荐复习的drill/card, 如'practice.md D4 Stage1 + schedule.json C4'>",
      "<推荐复习的notes.md段落, 如'notes.md 关键回顾3 宪法5维度表'>"
    ],
    "one_commitment": "<本周末前要完成的一个具体行动, 如'完成D4 Stage2填空+跑通deepeval HarmlessMetric 3个用例'>"
  }
}
```

**验收标准**:
- top_2_blind_spots 必须具体到 ILO 编号 (不能写"对齐不太熟")
- recommended_review_units 必须引用本单元真实文件 (practice.md/schedule.json/notes.md 的具体段落)
- one_commitment 必须可验证 (有明确交付物+截止时间)

### 与后续 Day 的衔接

- 本 tutorial 的 exit_artifact 中的盲点, 应在 Day 2 (AI安全威胁与防御) tutorial 前**已修复**
- Day 2 会复用本单元的"对齐评估"思路, 但聚焦 Prompt Injection / 红队测试
- 若 ILO4 (garak) 掌握度 <0.8, Day 2 的"红队测试"会困难 - 建议优先修复

---

*本 tutorial 引用 Oxford tutorial method (Socratic questioning, pre-tutorial essay) + HBS case method (devil's advocate) + Hattie & Timperley 4-level feedback (Feed Up/Back/Forward + Self-Reg 替代 Self 表扬)。静态 Socratic loop 不调用真实 LLM API, 用 if/else 分支模拟 Oxford tutor 的追问节奏。*
